# Lab 8.9 &mdash; Synthesis: One Coherent Reply

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Combine multiple agents' findings into one reply
- Keep it grounded -- built only from the findings
- Detect a conflict instead of pasting contradictions

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Synthesis — combine the parts](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-09"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Voting is for comparable **whole answers**; **synthesis** is for complementary **parts** (deck slide 14).
When the billing agent found the refund status and the tech agent diagnosed the bug, a **synthesiser** weaves
both into one coherent reply. Keys: **resolve conflicts** (don't just concatenate contradictions), **stay
grounded** (build only from what agents found), and **own one voice**.

In [ ]:
# findings: a dict of agent -> what it found.
print("synthesise:", {"billing": "duplicate charge -> refund", "tech": "BUG-231 -> update to v4.2"})

## Build it
Complete `synthesize` (combine, stable order) and `is_grounded` (every finding must appear).

In [ ]:
def synthesize(findings):
    # combine the findings, in a stable order, into ONE reply
    ordered = ___   # TODO: each finding's value, in sorted-key order
    return "Here is what we found: " + "; ".join(ordered) + "."

def is_grounded(reply, findings):
    # grounded = every finding actually appears in the reply (nothing dropped or invented)
    return ___   # TODO: True if every finding value is in the reply

def has_conflict(findings):
    joined = " ".join(findings.values()).lower()
    return "refund" in joined and "no refund" in joined

try:
    F = {"billing": "duplicate charge -> refund", "tech": "BUG-231 -> update to v4.2"}
    reply = synthesize(F)
    print(reply)
    print("grounded?", is_grounded(reply, F))
    print("conflict?", has_conflict({"a": "issue refund", "b": "no refund allowed"}))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The reply is **one string** in a **stable (sorted) order**, and `is_grounded` confirms **every** finding survived into it &mdash; nothing dropped or invented.
- A reply missing a finding fails `is_grounded` &mdash; the same grounding check the assembled chatbot uses so it never invents facts.
- `has_conflict` catches a *"refund"* vs *"no refund"* contradiction &mdash; a cue to **vote** (Lab 8.7) instead of pasting both.

## Your turn (open task &mdash; no grader)
Extend `synthesize` to **prefix each finding with its agent name** (e.g. `"billing: ..."`). **What good looks
like:** the reply still passes `is_grounded` (the finding text is intact), and now a reader can see *which*
specialist said *what* &mdash; the one-voice reply that the customer-service chatbot (Lab 8.11) will return.

---
### Key takeaway
Synthesis reconciles complementary parts into one grounded reply in a single voice -- the last step before the customer sees an answer. Vote to converge, critique to harden, synthesise to combine.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>